# RAG CoT
(Chain of Thought)

RAG 파이프라인에서 LLM이 단순한 정보 조합을 넘어서 단계적 사고를 통해 논리적 답변을 할 수 있도록 한다.

In [2]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
import os

load_dotenv()
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_PROJECT'] = 'skn34-langchain'
os.environ['LANGSMITH_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['PINECONE_API_KEY'] = os.getenv('PINECONE_API_KEY')

In [4]:
#가상 검색기
from langchain_core.documents import Document

def retrieve_vectordb(query = None):
    return [
        Document(page_content = '대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
        Document(page_content = '서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
        Document(page_content = '서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')
    ]

retrieve_vectordb()

[Document(metadata={}, page_content='대한민국의 수도는 서울입니다. 서울은 한강을 끼고 발달한 도시입니다.'),
 Document(metadata={}, page_content='서울의 대표적인 관광지는 경복궁, 남산타워, 명동 등이 있습니다.'),
 Document(metadata={}, page_content='서울의 인구는 약 천만 명이고, 교통 문화 인프라가 잘 갖추어져 있습니다.')]

In [5]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model('gpt-5.6-luna', temperature = 0)
prompt = ChatPromptTemplate.from_template('''
당신은 데이터를 분석해서 논리적인 결론을 도출하는 전문가 챗봇입니다.
아래 [검색된 문서]를 바탕으로 사용자의 [질문]에 대해 답변하세요.

[검색된 문서]
{context}

[질문]
{question}

[지시사항]
다음의 단계에 따라 사고한 후, 답변을 작성하세요.
1. **핵심데이터 정리**: 문서에서 사용자 질문과 관련한 팩트를 추출해보세요.
2. **상호관계 분석**: 각 항목별로 어떤 상관관계/시너지를 도출하는지 고민하세요.
3. **논리적 서술**: 위의 사고한 내용을 토대로 사용자 질문에 대한 답변을 준비하세요.
4. **최종 답변**: 서론-본론-결론 구조에 맞춰 완성된 답변을 작성하세요.
''')
output_parser = StrOutputParser()

chain = prompt | llm | output_parser

question = '서울의 인구, 관광지, 교통인프라를 종합해서 여행하기 좋은 이유를 논리적으로 설명해주세요.'
retrieved_docs = retrieve_vectordb(question)
# 문서 본문만 뽑아서 하나의 문자열 context
context = '\n\n'.join([doc.page_content for doc in retrieved_docs])

response = chain.invoke({'context': context, 'question': question})
print(response)

### 1. 핵심 데이터 정리

- **인구**: 서울의 인구는 약 1,000만 명입니다.
- **관광지**: 경복궁, 남산타워, 명동 등 대표적인 관광지가 있습니다.
- **교통·인프라**: 교통과 문화 인프라가 잘 갖추어져 있습니다.
- **도시 환경**: 한강을 끼고 발달한 도시입니다.

### 2. 상호관계 분석

서울은 인구가 많은 대도시이기 때문에 다양한 교통·문화 인프라가 발달해 있습니다. 이러한 인프라는 관광객이 여러 관광지를 편리하게 이동하고 이용할 수 있도록 돕습니다.

또한 경복궁에서는 역사와 전통문화를, 남산타워에서는 도시 경관을, 명동에서는 쇼핑과 도심 문화를 경험할 수 있습니다. 여기에 한강의 수변 환경까지 더해져 한 도시 안에서 다양한 여행 활동을 즐길 수 있습니다.

### 3. 논리적 서술

#### 서론

서울은 역사, 현대문화, 자연환경이 조화를 이루며 교통과 도시 인프라가 잘 갖추어진 도시입니다. 따라서 다양한 여행 목적을 충족하기에 적합한 관광지라고 볼 수 있습니다.

#### 본론

먼저 서울에는 경복궁, 남산타워, 명동과 같은 대표적인 관광지가 있어 역사 탐방, 도시 전망 감상, 쇼핑과 문화 체험을 모두 할 수 있습니다. 또한 한강을 끼고 발달한 도시이므로 도심 관광뿐 아니라 강을 중심으로 한 여가와 휴식도 기대할 수 있습니다.

서울의 약 1,000만 명에 이르는 인구는 도시의 규모와 다양성을 보여줍니다. 많은 사람이 생활하는 대도시인 만큼 교통과 문화 인프라가 잘 갖추어져 있어 관광객이 여러 지역과 명소를 비교적 편리하게 방문할 수 있습니다. 즉, 다양한 관광지가 존재하고 이를 연결하는 도시 기반시설이 뒷받침된다는 점에서 여행 효율성이 높습니다.

#### 결론

종합하면 서울은 역사적 명소, 현대적인 관광지, 한강의 도시 경관을 한곳에서 경험할 수 있으며, 잘 갖추어진 교통·문화 인프라 덕분에 관광지 간 이동과 여행 활동이 편리합니다. 따라서 다양한 볼거리와 편리한 여행 환경을 동시에 원하는 사람에게 서울은 여행하